## Splitting the input file based on the special character '@', and with every @ adding it to the next line for easy data handling

In [1]:
def new_line(input_file, output_file, symbol='@'):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile: # Opening the input file for reading and a new file to write the modified data in
        for line in infile:
            modified_line = line.replace(symbol, f'\n@') # Replacing @ symbol with @\n so that whenever an @ comes in the data, it will move to the next line
            outfile.write(modified_line)

input_file = 'Trump_Raw_Tweets.txt'
output_file = 'modified_new_line_trump_tweets.txt'
new_line(input_file, output_file)

## Data Cleaning

In [19]:
import re

def modify_func(line):
    val = line.split('\\x')[1:] # Splitting the data based on "\x" which is considered often to be the hexadecimal byte value in string
    for i in val: # Looping through each element
        byte_array = bytes(int(i, 16)) # Converting each hexadecimal string element into int and then converting it to an byte object
    try:
        return byte_array.decode('utf-8') # decoding the byte object into a UTF-8 string
    except UnicodeDecodeError: # Error Handling if the object cannot be decoded, then the original string will be returned
        return line

handling = r'\[0-9a-fA-F]{1,2}' # Pattern to check

start_list = []
with open('modified_new_line_trump_tweets.txt', 'r', encoding='utf-8') as file:
    for line in file:
        input_string = line.strip().replace("'b'RT", "").replace("b\'RT", "").replace("b\'", "") # Removing some special characters that were visible
        input_string = input_string.encode('utf-8').decode('unicode_escape') # Decoding escape sequences in the byte string back into their normal characters
        start_list.append(input_string) # Appending to a list


even_start_list = []
for input_string in start_list:
    output_string = re.sub(handling, modify_func, input_string) # Using the above defined function to clean up the data
    output_string = re.sub(r'\s?@\w+', '', output_string) # Using more regular expression to clean up the data
    output_string = re.sub(r"[\xa0\n]", " ", output_string)
    output_string = re.sub(r"[^a-zA-Z0-9\s+]", "", output_string)
    even_start_list.append(output_string.strip())


final_word_list = []
for i in even_start_list:
    x = i.split()
    for a in x:
        if a.startswith("https") or a == 'amp': # Removing list elements which starts with https or is amp
            continue
        else:
            final_word_list.append(a)


## Creating sets of positive, negative and other words to loop through (if there are any duplicates in the positive, negative and stop words, set will handle it)

In [22]:
positive = open("positive.txt","r")
positive_words = positive.read() # Opening and reading positive words
list_of_positive_words = positive_words.split() # Converting to list
list_of_positive_words = {i.lower() for i in list_of_positive_words} # making every element lower case
set_of_positive_words = set(list_of_positive_words) # Converting to set to remove any duplicate words

negative = open("negative.txt","r")
negative_words = negative.read() # Opening and reading negative words
list_of_negative_words = negative_words.split()
list_of_negative_words = {i.lower() for i in list_of_negative_words} # making every element lower case
set_of_negative_words = set(list_of_negative_words) # Converting to set to remove any duplicate words

stop = open("stopwords.txt","r")
stopwords = stop.read() # Opening and reading stop words
list_of_stopwords = stopwords.split()
list_of_stopwords = {i.lower() for i in list_of_stopwords} # making every element lower case
set_of_stopwords = set(list_of_stopwords) # Converting to set to remove any duplicate words


## Checking if the cleaned words are in the sets of positive, negative or stop words created and counting them

In [25]:
stop_cnt1 = 0
neg_cnt1 = 0
pos_cnt1 = 0
other_cnt1 = 0

for i in final_word_list:
    if i.lower() in set_of_positive_words:
        pos_cnt1 += 1
    elif i.lower() in set_of_negative_words:
        neg_cnt1 += 1
    elif i.lower() in set_of_stopwords:
        stop_cnt1 += 1
    else:
        other_cnt1 += 1

## Printing the final output and General Sentiment Analysis

In [15]:
print("Total Words Count =",len(final_word_list),"\n")

print("Positive Words Count =",pos_cnt1)
print("Negative Words Count =",neg_cnt1)
print("Stop Words Count =",stop_cnt1)
print("Other Words Count =",other_cnt1,"\n")

print("Positive to Total Words Ratio =",round((pos_cnt1/len(final_word_list)),6))
print("Negative to Total Words Ratio =",round((neg_cnt1/len(final_word_list)),6))
print("Stop Words to Total Ratio =",round(stop_cnt1/len(final_word_list),6))
print("Other Words to Total Ratio =",round(other_cnt1/len(final_word_list),6),"\n")

print("Positive vs Negative Word Count Ratio =",round(pos_cnt1/(neg_cnt1),6),"\n")


# General sentiment calculation

for_comp = round(pos_cnt1/(neg_cnt1),2)
if for_comp >= 2.0:
    print("General Sentiment of the analysis is 'Strongly Positive'")
elif for_comp < 2.0 and for_comp > 1.0:
    print("General Sentiment of the analysis is 'Weakly Positive'")
elif for_comp == 1.0:
    print("General Sentiment of the analysis is 'Neutral'")
elif for_comp < 1.0 and for_comp > 0.5:
    print("General Sentiment of the analysis is 'Weakly Negative'")
else:
    print("General Sentiment of the analysis is 'Strongly Negative'")

Total Words Count = 51428 

Positive Words Count = 3108
Negative Words Count = 2365
Stop Words Count = 21915
Other Words Count = 24040 

Positive to Total Words Ratio = 0.060434
Negative to Total Words Ratio = 0.045987
Stop Words to Total Ratio = 0.42613
Other Words to Total Ratio = 0.46745 

Positive vs Negative Word Count Ratio = 1.314165 

General Sentiment of the analysis is 'Weakly Positive'
